In [ ]:
from pathlib import Path
import random

# repo root = parent of the notebook folder
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
RAW_DIR = REPO_ROOT / "data" / "raw"

DOC_EXTS = {".md", ".mdx"}
SEED = 42
N = 4

random.seed(SEED)

print("cwd:", Path.cwd())
print("RAW_DIR resolved:", RAW_DIR.resolve())
print("RAW_DIR exists:", RAW_DIR.exists())

if not RAW_DIR.exists():
    raise SystemExit(f"RAW_DIR does not exist: {RAW_DIR}")

# --- collect all docs grouped by (library, version) ---
groups: dict[tuple[str, str], list[Path]] = {}
for p in sorted(RAW_DIR.rglob("*")):
    if p.is_file() and p.suffix in DOC_EXTS:
        rel = p.relative_to(RAW_DIR)
        if len(rel.parts) < 3:  # skip manifest.jsonl at the root
            continue
        library, version = rel.parts[0], rel.parts[1]
        groups.setdefault((library, version), []).append(p)

total = sum(len(v) for v in groups.values())
print(f"\nTotal docs found: {total}")
print("Groups found:")
for (lib, ver), files in groups.items():
    print(f"  {lib:10s} {ver:5s} -> {len(files)} files")

if total == 0:
    raise SystemExit("No docs found under RAW_DIR.")

# --- stratified pick: one per group, capped at N ---
keys = sorted(groups.keys())
random.shuffle(keys)

picked: list[Path] = []
for k in keys:
    if len(picked) >= N:
        break
    picked.append(random.choice(groups[k]))

# --- top up if we still need more ---
if len(picked) < N:
    picked_set = set(picked)
    pool = [p for files in groups.values() for p in files if p not in picked_set]
    need = min(N - len(picked), len(pool))
    if need > 0:
        picked += random.sample(pool, need)

print(f"\nSelected {len(picked)} files:")
for p in picked:
    print(f"  {p.relative_to(REPO_ROOT)}  ({p.stat().st_size:,} bytes)")

sample_files = picked
first_file = picked[0]

cwd: c:\Users\ASUS\OneDrive\Desktop\文件\Desktop\enterprise-rag\notebook
RAW_DIR resolved: C:\Users\ASUS\OneDrive\Desktop\文件\Desktop\enterprise-rag\data\raw
RAW_DIR exists: True

Total docs found: 150
Groups found:
  langchain  1.x   -> 61 files
  langgraph  0.6   -> 49 files
  langgraph  1.x   -> 40 files

Selected 4 files:
  data\raw\langgraph\0.6\agents\context.md  (13,210 bytes)
  data\raw\langchain\1.x\langchain\sql-agent.mdx  (20,328 bytes)
  data\raw\langgraph\1.x\langgraph\install.mdx  (1,371 bytes)
  data\raw\langgraph\0.6\agents\mcp.md  (13,809 bytes)


In [ ]:
from pathlib import Path
import re

def profile(p: Path) -> dict:
    text = p.read_text(encoding="utf-8", errors="replace")
    lines = text.splitlines()

    # front matter: must start on line 0 with ---
    has_front_matter = lines[:1] == ["---"]

    # conditional blocks
    n_python = len(re.findall(r"^:::python\b", text, flags=re.M))
    n_js     = len(re.findall(r"^:::js\b", text, flags=re.M))
    n_open   = len(re.findall(r"^:::\s*$", text, flags=re.M))  # closing ::: only
    n_other  = len(re.findall(r"^:::[a-zA-Z]", text, flags=re.M)) - n_python - n_js

    # mdx imports
    n_imports = len(re.findall(r"^\s*import\s+.*?from\s+", text, flags=re.M))

    # common JSX components
    jsx_hits = re.findall(r"<([A-Z][A-Za-z0-9_]*)\b", text)
    jsx_counts: dict[str, int] = {}
    for name in jsx_hits:
        jsx_counts[name] = jsx_counts.get(name, 0) + 1
    top_jsx = sorted(jsx_counts.items(), key=lambda kv: -kv[1])[:8]

    # structure
    n_headings = len(re.findall(r"^#{1,6}\s", text, flags=re.M))
    n_fences   = text.count("```") // 2  # pairs

    return {
        "file": p.relative_to(REPO_ROOT).as_posix(),
        "bytes": len(text),
        "front_matter": has_front_matter,
        "python_blocks": n_python,
        "js_blocks": n_js,
        "other_directives": n_other,
        "close_markers": n_open,
        "mdx_imports": n_imports,
        "headings": n_headings,
        "code_fences": n_fences,
        "top_jsx": top_jsx,
    }

for p in sample_files:
    info = profile(p)
    print("=" * 78)
    print(info["file"], f"({info['bytes']:,} bytes)")
    print(f"  front_matter={info['front_matter']}  python_blocks={info['python_blocks']}  "
          f"js_blocks={info['js_blocks']}  other_directives={info['other_directives']}  "
          f"close_markers={info['close_markers']}")
    print(f"  mdx_imports={info['mdx_imports']}  headings={info['headings']}  code_fences={info['code_fences']}")
    print(f"  top JSX: {info['top_jsx']}")

data/raw/langgraph/0.6/agents/context.md (12,902 bytes)
  front_matter=False  python_blocks=1  js_blocks=1  other_directives=0  close_markers=2
  mdx_imports=7  headings=5  code_fences=9
  top JSX: []
data/raw/langchain/1.x/langchain/sql-agent.mdx (19,833 bytes)
  front_matter=True  python_blocks=9  js_blocks=8  other_directives=0  close_markers=17
  mdx_imports=23  headings=5  code_fences=17
  top JSX: [('Step', 9), ('Note', 4), ('Warning', 2), ('Steps', 2), ('CodeGroup', 2), ('Accordion', 2), ('ChatModelTabsPy', 1), ('ChatModelTabsJS', 1)]
data/raw/langgraph/1.x/langgraph/install.mdx (1,287 bytes)
  front_matter=True  python_blocks=2  js_blocks=2  other_directives=0  close_markers=4
  mdx_imports=0  headings=2  code_fences=12
  top JSX: [('CodeGroup', 4)]
data/raw/langgraph/0.6/agents/mcp.md (13,292 bytes)
  front_matter=True  python_blocks=6  js_blocks=6  other_directives=0  close_markers=10
  mdx_imports=14  headings=5  code_fences=12
  top JSX: []


In [ ]:
for p in sample_files:
    text = p.read_text(encoding="utf-8", errors="replace")
    print("=" * 78)
    print(p.relative_to(REPO_ROOT).as_posix())
    print("-" * 78)
    print(text[:1800])
    print("...\n")

data/raw/langgraph/0.6/agents/context.md
------------------------------------------------------------------------------
# Context

**Context engineering** is the practice of building dynamic systems that provide the right information and tools, in the right format, so that an AI application can accomplish a task. Context can be characterized along two key dimensions:

1. By **mutability**:
    - **Static context**: Immutable data that doesn't change during execution (e.g., user metadata, database connections, tools)
    - **Dynamic context**: Mutable data that evolves as the application runs (e.g., conversation history, intermediate results, tool call observations)
2. By **lifetime**:
    - **Runtime context**: Data scoped to a single run or invocation
    - **Cross-conversation context**: Data that persists across multiple conversations or sessions

!!! tip "Runtime context vs LLM context"

    Runtime context refers to local context: data and dependencies your code needs to run. It d

In [ ]:
# after building `groups` above, before picking:
current_files = [p for p in groups.get(("langchain", "1.x"), [])] \
              + [p for p in groups.get(("langgraph", "1.x"), [])]

# always include two random current files, then top up to N
random.seed(SEED)
picked = random.sample(current_files, 2)
picked_set = set(picked)

# one from each remaining group
for (lib, ver), files in groups.items():
    if len(picked) >= N:
        break
    choices = [f for f in files if f not in picked_set]
    if choices:
        c = random.choice(choices)
        picked.append(c)
        picked_set.add(c)

# top up if needed
if len(picked) < N:
    pool = [p for files in groups.values() for p in files if p not in picked_set]
    need = min(N - len(picked), len(pool))
    picked += random.sample(pool, need)

print(f"\nSelected {len(picked)} files:")
for p in picked:
    print(f"  {p.relative_to(REPO_ROOT)}  ({p.stat().st_size:,} bytes)")

sample_files = picked


Selected 4 files:
  data\raw\langgraph\1.x\langgraph\observability.mdx  (6,626 bytes)
  data\raw\langchain\1.x\langchain\errors\MODEL_RATE_LIMIT.mdx  (1,127 bytes)
  data\raw\langchain\1.x\concepts\memory.mdx  (21,872 bytes)
  data\raw\langgraph\0.6\troubleshooting\errors\MULTIPLE_SUBGRAPHS.md  (828 bytes)


In [ ]:
p = REPO_ROOT / "data/raw/langchain/1.x/concepts/memory.mdx"
text = p.read_text(encoding="utf-8", errors="replace")
print(text[:2500])

---
title: Memory overview
sidebarTitle: Memory
---



[Memory](/oss/langgraph/add-memory) is a system that remembers information about previous interactions. For AI agents, memory is crucial because it lets them remember previous interactions, learn from feedback, and adapt to user preferences. As agents tackle more complex tasks with numerous user interactions, this capability becomes essential for both efficiency and user satisfaction.

This conceptual guide covers two types of memory, based on their recall scope:

* [Short-term memory](#short-term-memory), or [thread](/oss/langgraph/checkpointers#threads)-scoped memory, tracks the ongoing conversation by maintaining message history within a session. LangGraph manages short-term memory as a part of your agent's [state](/oss/langgraph/graph-api#state). State is persisted to a database using a [checkpointer](/oss/langgraph/checkpointers#checkpoints) so the thread can be resumed at any time. Short-term memory updates when the graph is 

In [ ]:
import re
from pathlib import Path

MDX_SIGNALS = [
    ("front_matter",  r"\A---\s*\n"),
    ("mdx_import",    r"(?m)^\s*import\s+.+?from\s+"),
    ("colon_block",   r"(?m)^:::[a-zA-Z]"),
    ("jsx_tag",       r"<[A-Z][A-Za-z0-9_]*[\s/>]"),
    ("codegroup",     r"<CodeGroup>"),
    ("mkdocs_am",     r"(?m)^!!!\s+\w+"),
]

for p in sample_files:
    text = p.read_text(encoding="utf-8", errors="replace")
    hits = {name: len(re.findall(pat, text)) for name, pat in MDX_SIGNALS}
    fmt = "MDX" if (hits["colon_block"] or hits["mdx_import"] or hits["jsx_tag"]) else "MKDOCS"
    print(f"{p.relative_to(REPO_ROOT).as_posix():55s} {fmt:8s} {hits}")

data/raw/langgraph/1.x/langgraph/observability.mdx      MDX      {'front_matter': 1, 'mdx_import': 8, 'colon_block': 8, 'jsx_tag': 3, 'codegroup': 0, 'mkdocs_am': 0}
data/raw/langchain/1.x/langchain/errors/MODEL_RATE_LIMIT.mdx MDX      {'front_matter': 1, 'mdx_import': 0, 'colon_block': 0, 'jsx_tag': 1, 'codegroup': 0, 'mkdocs_am': 0}
data/raw/langchain/1.x/concepts/memory.mdx              MDX      {'front_matter': 1, 'mdx_import': 1, 'colon_block': 8, 'jsx_tag': 1, 'codegroup': 0, 'mkdocs_am': 0}
data/raw/langgraph/0.6/troubleshooting/errors/MULTIPLE_SUBGRAPHS.md MDX      {'front_matter': 0, 'mdx_import': 0, 'colon_block': 2, 'jsx_tag': 0, 'codegroup': 0, 'mkdocs_am': 0}


In [ ]:
import re
from pathlib import Path

# ---------- regexes ----------
_FRONT_MATTER = re.compile(r"\A---\s*\n(.*?)\n---\s*\n", re.S)
_IMPORT = re.compile(r"(?m)^\s*import\s+[^\n]*?from\s+[^\n]*?;?\s*$")
_JSX_SELF = re.compile(r"<[A-Z][A-Za-z0-9_]*(?:\s[^<>]*?)?/\s*>")
_JSX_OPEN = re.compile(r"<[A-Z][A-Za-z0-9_]*(?:\s[^<>]*?)?>")
_JSX_CLOSE = re.compile(r"</[A-Z][A-Za-z0-9_]*\s*>")
_IMG = re.compile(r"!\[[^\]]*\]\([^)]*\)")
_BLANK3 = re.compile(r"\n{3,}")
_COLON = re.compile(r"^:::([a-zA-Z][A-Za-z0-9_-]*)\s*$")

DROP_LANGS = {"js", "javascript", "ts", "typescript", "jsx", "tsx"}
KEEP_LANGS = {"python", "py"}


def _parse_front_matter(text):
    m = _FRONT_MATTER.match(text)
    if not m:
        return {}, text
    meta = {}
    for line in m.group(1).splitlines():
        if ":" in line:
            k, _, v = line.partition(":")
            meta[k.strip()] = v.strip().strip('"').strip("'")
    return meta, text[m.end():]


def _resolve_colon_blocks(text, keep="python"):
    """Handle :::python / :::js (and admonitions like :::note)."""
    lines = text.splitlines()
    out, i, n = [], 0, len(lines)
    stats = {"kept_python": 0, "dropped_js": 0, "kept_other": 0}
    while i < n:
        m = _COLON.match(lines[i])
        if m:
            lang = m.group(1).lower()
            j = i + 1
            body = []
            while j < n and lines[j].strip() != ":::":
                body.append(lines[j]); j += 1
            if lang in DROP_LANGS:
                stats["dropped_js"] += 1
            elif lang in KEEP_LANGS:
                out.extend(body); stats["kept_python"] += 1
            else:
                out.extend(body); stats["kept_other"] += 1
            i = j + 1 if j < n else n
            continue
        out.append(lines[i]); i += 1
    return "\n".join(out), stats


def clean_mdx(text, keep_lang="python"):
    """Mintlify MDX (current docs, langchain-ai/docs@main)."""
    meta, body = _parse_front_matter(text)
    body = _IMPORT.sub("", body)
    body, stats = _resolve_colon_blocks(body, keep=keep_lang)
    body = _JSX_SELF.sub("", body)
    body = _JSX_OPEN.sub("", body)
    body = _JSX_CLOSE.sub("", body)
    body = _IMG.sub("", body)
    body = _BLANK3.sub("\n\n", body).strip() + "\n"
    return body, meta, stats


def clean_mkdocs(text):
    """MkDocs Material (LangGraph 0.6). Convert !!! admonitions to blockquotes
    so 4-space indentation isn't mistaken for code blocks downstream."""
    lines = text.splitlines()
    out, i, n = [], 0, len(lines)
    converted = 0
    while i < n:
        m = re.match(r"^!!!\s+(\w+)(?:\s+\"([^\"]*)\")?\s*$", lines[i])
        if m:
            kind, title = m.group(1), m.group(2)
            header = f"> **{kind.title()}"
            if title:
                header += f": {title}"
            header += "**"
            out.append(header)
            converted += 1
            i += 1
            while i < n and (not lines[i].strip() or lines[i].startswith("    ")):
                body_line = lines[i][4:] if lines[i].startswith("    ") else ""
                out.append("> " + body_line if body_line.strip() else ">")
                i += 1
            continue
        out.append(lines[i]); i += 1
    return "\n".join(out), {}, {"converted_admonitions": converted}


def clean_file(path: Path, keep_lang="python"):
    raw = path.read_text(encoding="utf-8", errors="replace")
    is_mdx = bool(
        re.search(r"(?m)^\s*import\s+.*from\s+", raw)
        or re.search(r"(?m)^:::[a-zA-Z]", raw)
        or re.search(r"<[A-Z][A-Za-z0-9_]*[\s/>]", raw)
        or path.suffix == ".mdx"
    )
    if is_mdx:
        cleaned, meta, stats = clean_mdx(raw, keep_lang=keep_lang)
        return cleaned, meta, stats, "MDX"
    cleaned, meta, stats = clean_mkdocs(raw)
    return cleaned, meta, stats, "MKDOCS"

In [24]:
for p in sample_files:
    cleaned, meta, stats, fmt = clean_file(p)
    raw = p.read_text(encoding="utf-8", errors="replace")

    print("=" * 78)
    print(f"{p.relative_to(REPO_ROOT).as_posix()}  [{fmt}]")
    print(f"  meta: {meta}")
    print(f"  stats: {stats}")
    print(f"  bytes: {len(raw):,} -> {len(cleaned):,}")

    checks = {
        "front_matter_left": bool(re.match(r"\A---\s*\n", cleaned)),
        "mdx_import_left": len(re.findall(r"(?m)^\s*import\s+.*from\s+", cleaned)),
        "colon_block_left": len(re.findall(r"(?m)^:::[a-zA-Z]", cleaned)),
        "jsx_left": len(re.findall(r"<[A-Z][A-Za-z0-9_]*[\s/>]", cleaned)),
        "headings_before": len(re.findall(r"(?m)^#{1,6}\s", raw)),
        "headings_after": len(re.findall(r"(?m)^#{1,6}\s", cleaned)),
        "fences_before": raw.count("```"),
        "fences_after": cleaned.count("```"),
    }
    print(f"  checks: {checks}")

data/raw/langgraph/1.x/langgraph/observability.mdx  [MDX]
  meta: {'title': 'LangSmith Observability'}
  stats: {'kept_python': 4, 'dropped_js': 4, 'kept_other': 0}
  bytes: 6,402 -> 4,320
  checks: {'front_matter_left': False, 'mdx_import_left': 0, 'colon_block_left': 0, 'jsx_left': 0, 'headings_before': 9, 'headings_after': 9, 'fences_before': 22, 'fences_after': 14}
data/raw/langchain/1.x/langchain/errors/MODEL_RATE_LIMIT.mdx  [MDX]
  meta: {'title': 'MODEL_RATE_LIMIT'}
  stats: {'kept_python': 0, 'dropped_js': 0, 'kept_other': 0}
  bytes: 1,104 -> 1,052
  checks: {'front_matter_left': False, 'mdx_import_left': 0, 'colon_block_left': 0, 'jsx_left': 0, 'headings_before': 1, 'headings_after': 1, 'fences_before': 0, 'fences_after': 0}
data/raw/langchain/1.x/concepts/memory.mdx  [MDX]
  meta: {'title': 'Memory overview', 'sidebarTitle': 'Memory'}
  stats: {'kept_python': 4, 'dropped_js': 4, 'kept_other': 0}
  bytes: 21,589 -> 17,786
  checks: {'front_matter_left': False, 'mdx_import_lef

In [25]:
p = sample_files[0]
cleaned, meta, stats, fmt = clean_file(p)
print(cleaned[:2000])

Traces are a series of steps that your application takes to go from input to output. Each of these individual steps is represented by a run. You can use [LangSmith](https://smith.langchain.com) to visualize these execution steps. To use it, [enable tracing for your application](/langsmith/trace-with-langgraph). This enables you to do the following:

* [Debug a locally running application](/langsmith/observability-studio#debug-langsmith-traces).
* [Evaluate the application performance](/oss/langchain/test/evals).
* [Monitor the application](/langsmith/dashboards).

## Prerequisites

Before you begin, ensure you have the following:

- **A LangSmith account**: Sign up (for free) or log in at [smith.langchain.com](https://smith.langchain.com).
- **A LangSmith API key**: Follow the [Create an API key](/langsmith/create-account-api-key) guide.

## Enable tracing

To enable tracing for your application, set the following environment variables:

```python
export LANGSMITH_TRACING=true
export L

In [26]:
from pathlib import Path
import json, hashlib

OUT_DIR = REPO_ROOT / "data" / "processed"
OUT_DIR.mkdir(parents=True, exist_ok=True)

manifest = (REPO_ROOT / "data" / "raw" / "manifest.jsonl").read_text().splitlines()
rows = [json.loads(l) for l in manifest if l.strip()]

results = []
failures = []
for row in rows:
    src = REPO_ROOT / row["local_path"]
    try:
        cleaned, meta, stats, fmt = clean_file(src)
        dest = OUT_DIR / row["library"] / row["version"] / Path(row["source_path"]).with_suffix(".md")
        dest.parent.mkdir(parents=True, exist_ok=True)
        dest.write_text(cleaned, encoding="utf-8")
        results.append({
            **row,
            "format": fmt,
            "cleaned_path": str(dest.relative_to(REPO_ROOT)),
            "cleaned_bytes": len(cleaned),
            "cleaned_sha256": hashlib.sha256(cleaned.encode()).hexdigest(),
            "front_matter": meta,
            "stats": stats,
        })
    except Exception as e:
        failures.append({"path": row["local_path"], "error": str(e)})

(OUT_DIR / "cleaned_manifest.jsonl").write_text(
    "\n".join(json.dumps(r) for r in results) + "\n"
)
print(f"cleaned={len(results)} failed={len(failures)}")
for f in failures[:10]:
    print("  FAIL", f)

cleaned=150 failed=0


In [27]:
import json, re
from pathlib import Path

OUT_DIR = REPO_ROOT / "data" / "processed"
manifest = [json.loads(l) for l in (OUT_DIR / "cleaned_manifest.jsonl").read_text().splitlines() if l.strip()]

problems = []
summary = {"MDX": 0, "MKDOCS": 0}
for r in manifest:
    cleaned = (REPO_ROOT / r["cleaned_path"]).read_text(encoding="utf-8", errors="replace")
    raw     = (REPO_ROOT / r["local_path"]).read_text(encoding="utf-8", errors="replace")

    summary[r["format"]] = summary.get(r["format"], 0) + 1

    issues = []
    if re.match(r"\A---\s*\n", cleaned):
        issues.append("front_matter_left")
    if len(re.findall(r"(?m)^\s*import\s+.*from\s+", cleaned)) > 0:
        issues.append("mdx_import_left")
    if len(re.findall(r"(?m)^:::[a-zA-Z]", cleaned)) > 0:
        issues.append("colon_block_left")
    if len(re.findall(r"<[A-Z][A-Za-z0-9_]*[\s/>]", cleaned)) > 0:
        issues.append("jsx_left")

    h_before = len(re.findall(r"(?m)^#{1,6}\s", raw))
    h_after  = len(re.findall(r"(?m)^#{1,6}\s", cleaned))
    if h_before != h_after:
        issues.append(f"headings_lost({h_before}->{h_after})")

    f_before = raw.count("```")
    f_after  = cleaned.count("```")
    if f_before != f_after:
        issues.append(f"fences_changed({f_before}->{f_after})")

    if issues:
        problems.append((r["cleaned_path"], issues))

print(f"formats: {summary}")
print(f"files with issues: {len(problems)} / {len(manifest)}")
for path, issues in problems[:30]:
    print(f"  {path}  ->  {issues}")

formats: {'MDX': 138, 'MKDOCS': 12}
files with issues: 119 / 150
  data\processed\langchain\1.x\langchain\agents.md  ->  ['fences_changed(30->12)']
  data\processed\langchain\1.x\langchain\context-engineering.md  ->  ['fences_changed(92->86)']
  data\processed\langchain\1.x\langchain\deep-agent-from-scratch.md  ->  ['fences_changed(30->26)']
  data\processed\langchain\1.x\langchain\errors\INVALID_PROMPT_INPUT.md  ->  ['fences_changed(4->0)']
  data\processed\langchain\1.x\langchain\errors\INVALID_TOOL_RESULTS.md  ->  ['fences_changed(4->2)']
  data\processed\langchain\1.x\langchain\errors\MODEL_AUTHENTICATION.md  ->  ['fences_changed(4->2)']
  data\processed\langchain\1.x\langchain\event-streaming.md  ->  ['fences_changed(44->24)']
  data\processed\langchain\1.x\langchain\guardrails.md  ->  ['fences_changed(24->14)']
  data\processed\langchain\1.x\langchain\human-in-the-loop.md  ->  ['fences_changed(34->22)']
  data\processed\langchain\1.x\langchain\install.md  ->  ['fences_changed(24-

In [28]:
import random
random.seed(7)
for r in random.sample(manifest, 3):
    cleaned = (REPO_ROOT / r["cleaned_path"]).read_text(encoding="utf-8", errors="replace")
    print("=" * 78)
    print(r["cleaned_path"], f"[{r['format']}]")
    print("-" * 78)
    print(cleaned[:1200])
    print()

data\processed\langgraph\1.x\langgraph\test.md [MDX]
------------------------------------------------------------------------------
After you've prototyped your LangGraph agent, a natural next step is to add tests. This guide covers some useful patterns you can use when writing unit tests.

Note that this guide is LangGraph-specific and covers scenarios around graphs with custom structures - if you are just getting started, check out [Test](/oss/langchain/test/) that uses LangChain's built-in @[`create_agent`] instead.

## Prerequisites

First, make sure you have [`pytest`](https://docs.pytest.org/) installed:

```bash
$ pip install -U pytest
```

## Getting started

Because many LangGraph agents depend on state, a useful pattern is to create your graph before each test where you use it, then compile it within tests with a new checkpointer instance.

The below example shows how this works with a simple, linear graph that progresses through `node1` and `node2`. Each node updates the sin

In [29]:
import sys
import importlib
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name in {"notebook", "notebooks"} else Path.cwd()
SRC = REPO_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import rag.ingestion.chunkers as ch
importlib.reload(ch)
from rag.ingestion.chunkers import chunk_document

from llama_index.core.schema import Document, NodeRelationship

# sample_files already defined in an earlier cell
for p in sample_files:
    rel = p.relative_to(REPO_ROOT).parts  # ('data','raw','<lib>','<ver>','...', filename)
    library, version = rel[2], rel[3]
    source_path = "/".join(rel[4:])

    doc = Document(
        text=p.read_text(encoding="utf-8", errors="replace"),
        metadata={
            "library": library,
            "version": version,
            "source_path": source_path,
        },
    )
    # deterministic doc_id so leaves keep a stable SOURCE pointer
    from rag.ingestion.ids import make_doc_id
    doc.doc_id = make_doc_id(library, version, source_path)

    parents, leaves = chunk_document(doc)

    print("=" * 78)
    print(f"{library}/{version}/{source_path}")
    print(f"  parents={len(parents)}  leaves={len(leaves)}")
    if parents:
        sizes = [p.metadata["token_count"] for p in parents]
        print(f"  parent tokens: min={min(sizes)} max={max(sizes)}")
    if leaves:
        sizes = [l.metadata["token_count"] for l in leaves]
        print(f"  leaf tokens:   min={min(sizes)} max={max(sizes)}")
        print(f"  leaves with code: {sum(1 for l in leaves if l.metadata['has_code'])}")

    # --- assertions ---
    for node in leaves:
        # code fences must be balanced (count of ``` must be even)
        fence_count = node.text.count("```") + node.text.count("~~~")
        assert fence_count % 2 == 0, f"odd fence count in leaf {node.id_}"

        # every leaf points to its parent
        assert NodeRelationship.PARENT in node.relationships, node.id_

    for i, node in enumerate(leaves):
        if i > 0:
            assert NodeRelationship.PREVIOUS in node.relationships, node.id_
        if i < len(leaves) - 1:
            assert NodeRelationship.NEXT in node.relationships, node.id_

print("\nall assertions passed")

langgraph/1.x/langgraph/observability.mdx
  parents=8  leaves=9
  parent tokens: min=70 max=401
  leaf tokens:   min=74 max=255
  leaves with code: 6


AssertionError: odd fence count in leaf 42ac6b1117c700e68b604ddf7805457f

In [ ]:
# find the failing leaf and print its text
for p in sample_files:
    rel = p.relative_to(REPO_ROOT).parts
    library, version = rel[2], rel[3]
    source_path = "/".join(rel[4:])
    doc = Document(
        text=p.read_text(encoding="utf-8", errors="replace"),
        metadata={"library": library, "version": version, "source_path": source_path},
    )
    from rag.ingestion.ids import make_doc_id
    doc.doc_id = make_doc_id(library, version, source_path)
    _, leaves = chunk_document(doc)

    for node in leaves:
        if node.text.count("```") % 2 != 0:
            print("=" * 78)
            print(f"file: {source_path}")
            print(f"leaf id: {node.id_}")
            print(f"heading: {node.metadata.get('heading_path')}")
            print("-" * 78)
            print(node.text)
            print("-" * 78)

# also: search source files for non-plain fence lines
import re
FENCE_PROBE = re.compile(r"^(`{3,}|~{3,})(\s+\S.*)?$")
for p in sample_files:
    text = p.read_text(encoding="utf-8", errors="replace")
    for i, line in enumerate(text.splitlines(), 1):
        m = FENCE_PROBE.match(line)
        if m and m.group(2):
            print(f"{p.name}:{i}: {line[:80]!r}")

file: langgraph/observability.mdx
leaf id: 42ac6b1117c700e68b604ddf7805457f
heading: Use anonymizers to prevent logging of sensitive data in traces
------------------------------------------------------------------------------
[Use anonymizers to prevent logging of sensitive data in traces]

## Use anonymizers to prevent logging of sensitive data in traces

You may want to mask sensitive data to prevent it from being logged to LangSmith. You can create [anonymizers](/langsmith/mask-inputs-outputs#rule-based-masking-of-inputs-and-outputs) and apply them to
your graph using configuration. This example will redact anything matching the Social Security Number format XXX-XX-XXXX from traces sent to LangSmith.

:::python
```python Python
from langchain_core.tracers.langchain import LangChainTracer
from langgraph.graph import StateGraph, MessagesState
from langsmith import Client
from langsmith.anonymizer import create_anonymizer

anonymizer = create_anonymizer([
    # Matches SSNs
    { "pat

In [ ]:
import json, importlib, random
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name in {"notebook", "notebooks"} else Path.cwd()
PROCESSED = REPO_ROOT / "data" / "processed"

import rag.ingestion.chunkers as ch
importlib.reload(ch)
from rag.ingestion.chunkers import chunk_document
from rag.ingestion.ids import make_doc_id
from llama_index.core.schema import Document, NodeRelationship

# --- pick 4 cleaned files, stratified by (library, version) ---
manifest = [json.loads(l) for l in (PROCESSED / "cleaned_manifest.jsonl").read_text().splitlines() if l.strip()]

groups: dict[tuple[str, str], list[dict]] = {}
for r in manifest:
    groups.setdefault((r["library"], r["version"]), []).append(r)

random.seed(42)
picked_rows = [random.choice(v) for v in groups.values()]
# top up to 4 if fewer groups
if len(picked_rows) < 4:
    pool = [r for v in groups.values() for r in v if r not in picked_rows]
    picked_rows += random.sample(pool, min(4 - len(picked_rows), len(pool)))

# --- sanity: no MDX noise left in cleaned files ---
import re
def mdx_noise(text: str) -> dict:
    return {
        "colon": len(re.findall(r"(?m)^:::[a-zA-Z]", text)),
        "import": len(re.findall(r"(?m)^\s*import\s+.*from\s+", text)),
        "jsx": len(re.findall(r"<[A-Z][A-Za-z0-9_]*[\s/>]", text)),
    }

_FENCE_LINE = re.compile(r"^(`{3,}|~{3,})", re.M)
def fence_balanced(text: str) -> bool:
    return len(_FENCE_LINE.findall(text)) % 2 == 0

for r in picked_rows:
    cp = REPO_ROOT / r["cleaned_path"]
    text = cp.read_text(encoding="utf-8", errors="replace")

    noise = mdx_noise(text)
    print("=" * 78)
    print(f"{r['library']}/{r['version']}/{r['source_path']}  [{r['format']}]")
    print(f"  noise: {noise}")
    if any(noise.values()):
        print(f"  ^ cleaning didn't fully process this file")

    doc = Document(
        text=text,
        metadata={
            "library": r["library"],
            "version": r["version"],
            "source_path": r["source_path"],
        },
    )
    doc.doc_id = make_doc_id(r["library"], r["version"], r["source_path"])

    parents, leaves = chunk_document(doc)
    print(f"  parents={len(parents)}  leaves={len(leaves)}")
    if parents:
        s = [p.metadata["token_count"] for p in parents]
        print(f"  parent tokens: min={min(s)} max={max(s)}")
    if leaves:
        s = [l.metadata["token_count"] for l in leaves]
        print(f"  leaf tokens:   min={min(s)} max={max(s)}")
        print(f"  leaves w/ code: {sum(1 for l in leaves if l.metadata['has_code'])}")

    for node in leaves:
        assert fence_balanced(node.text), f"unbalanced fence in {node.id_}"
        assert NodeRelationship.PARENT in node.relationships, node.id_
    for i, node in enumerate(leaves):
        if i > 0:
            assert NodeRelationship.PREVIOUS in node.relationships, node.id_
        if i < len(leaves) - 1:
            assert NodeRelationship.NEXT in node.relationships, node.id_

print("\nall assertions passed")

langchain/1.x/langchain/quickstart.mdx  [MDX]
  noise: {'colon': 0, 'import': 0, 'jsx': 0}
  parents=19  leaves=46
  parent tokens: min=37 max=1225
  leaf tokens:   min=37 max=310
  leaves w/ code: 1


AssertionError: unbalanced fence in 543fa756c2d8853415fb708d61a11a8e

In [30]:
import re
import importlib
import rag.ingestion.chunkers as ch
importlib.reload(ch)
from rag.ingestion.chunkers import chunk_document, parse_blocks, _FENCE_RE
from rag.ingestion.ids import make_doc_id
from llama_index.core.schema import Document

print("1) is the regex fix applied?")
print("   _FENCE_RE pattern:", _FENCE_RE.pattern)
print("   -> should be: ^(`{3,}|~{3,})\\s*(.*?)\\s*$")
print()

# quickstart file
row = next(r for r in manifest if r["source_path"].endswith("quickstart.mdx")
           and r["version"] == "1.x")

cp = REPO_ROOT / row["cleaned_path"]
text = cp.read_text(encoding="utf-8", errors="replace")

print("2) every line in cleaned quickstart that looks like a fence:")
for i, line in enumerate(text.splitlines(), 1):
    if re.match(r"^(`{3,}|~{3,})", line):
        print(f"   {i:4d}: {line[:90]!r}")
print()

print("3) what parse_blocks sees (kind, lang, first 60 chars):")
blocks = parse_blocks(text)
from collections import Counter
kinds = Counter(b.kind for b in blocks)
print("   block kinds:", dict(kinds))
print("   code blocks:", sum(1 for b in blocks if b.kind == "code"))
for b in blocks:
    if b.kind == "code":
        print(f"     lang={b.lang!r}  head={b.text[:60]!r}")
print()

print("4) the failing leaf content:")
doc = Document(text=text, metadata={
    "library": row["library"], "version": row["version"],
    "source_path": row["source_path"],
})
doc.doc_id = make_doc_id(row["library"], row["version"], row["source_path"])
_, leaves = chunk_document(doc)
_FENCE_LINE = re.compile(r"^(`{3,}|~{3,})", re.M)
bad = [l for l in leaves if len(_FENCE_LINE.findall(l.text)) % 2 != 0]
print(f"   {len(bad)} leaf/leaves with odd fence count")
for l in bad[:2]:
    print("   ---")
    print("   id:", l.id_)
    print("   fence lines:", [ln for ln in l.text.splitlines() if _FENCE_LINE.match(ln)])
    print("   full text:")
    for ln in l.text.splitlines():
        print("     ", ln)

1) is the regex fix applied?
   _FENCE_RE pattern: ^(`{3,}|~{3,})\s*(\S*)\s*$
   -> should be: ^(`{3,}|~{3,})\s*(.*?)\s*$

2) every line in cleaned quickstart that looks like a fence:
   1201: '```shell'
   1204: '```'

3) what parse_blocks sees (kind, lang, first 60 chars):
   block kinds: {'para': 213, 'heading': 12, 'list': 12, 'code': 1}
   code blocks: 1
     lang='shell'  head='```shell\nexport LANGSMITH_TRACING="true"\nexport LANGSMITH_AP'

4) the failing leaf content:
   3 leaf/leaves with odd fence count
   ---
   id: 543fa756c2d8853415fb708d61a11a8e
   fence lines: ['```bash Shell']
   full text:
      ```bash Shell
          # Local: Ollama must be running (https://ollama.com)
          # Cloud: Set your Ollama API key for hosted inference
          export OLLAMA_API_KEY="your-api-key"
          ```
      
          ```bash .env
          # Local: Ollama must be running (https://ollama.com)
          # Cloud: Set your Ollama API key for hosted inference
          OLLAMA_API_

In [32]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name in {"notebook", "notebooks"} else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "src"))

import importlib
import rag.ingestion.chunkers as ch
importlib.reload(ch)

print("_FENCE_RE.pattern   =", ch._FENCE_RE.pattern)
print("_HEADING_RE.pattern =", ch._HEADING_RE.pattern)
print()

expected_fence   = r"^[ ]{0,6}(`{3,}|~{3,})[ \t]*(.*?)[ \t]*$"
expected_heading = r"^[ ]{0,3}(#{1,6})\s+(.*?)\s*$"

ok_fence   = ch._FENCE_RE.pattern == expected_fence
ok_heading = ch._HEADING_RE.pattern == expected_heading

print("fence regex updated:  ", ok_fence)
print("heading regex updated:", ok_heading)

if not (ok_fence and ok_heading):
    print()
    print("!! File edits did NOT take effect. Check:")
    print("   - chunkers.py saved to", (REPO_ROOT / 'src' / 'rag' / 'ingestion' / 'chunkers.py'))
    print("   - Kernel -> Restart, then re-run this cell")
else:
    # also verify Section.text no longer strips leading whitespace
    from rag.ingestion.chunkers import Section, Block
    s = Section(heading_path="x", heading_path_list=["x"], section_title="x",
                blocks=[Block(kind="code", text="    ```py\n    x=1\n    ```")])
    print("Section.text preserves indent:", s.text.startswith("    ```"))

_FENCE_RE.pattern   = ^(`{3,}|~{3,})\s*(\S*)\s*$
_HEADING_RE.pattern = ^(#{1,6})\s+(.*?)\s*$

fence regex updated:   False
heading regex updated: False

!! File edits did NOT take effect. Check:
   - chunkers.py saved to c:\Users\ASUS\OneDrive\Desktop\文件\Desktop\enterprise-rag\src\rag\ingestion\chunkers.py
   - Kernel -> Restart, then re-run this cell


In [33]:
import json, random, re
from pathlib import Path

import importlib
import rag.ingestion.chunkers as ch
importlib.reload(ch)
from rag.ingestion.chunkers import chunk_document
from rag.ingestion.ids import make_doc_id
from llama_index.core.schema import Document, NodeRelationship

REPO_ROOT = Path.cwd().parent if Path.cwd().name in {"notebook", "notebooks"} else Path.cwd()
PROCESSED = REPO_ROOT / "data" / "processed"

# --- load cleaned manifest ---
manifest = [
    json.loads(l)
    for l in (PROCESSED / "cleaned_manifest.jsonl").read_text().splitlines()
    if l.strip()
]

# --- stratified pick: one per (library, version), top up to 4 ---
groups: dict[tuple[str, str], list[dict]] = {}
for r in manifest:
    groups.setdefault((r["library"], r["version"]), []).append(r)

random.seed(42)
picked = [random.choice(v) for v in groups.values()]
if len(picked) < 4:
    seen = {id(r) for r in picked}
    pool = [r for v in groups.values() for r in v if id(r) not in seen]
    picked += random.sample(pool, min(4 - len(picked), len(pool)))

# --- helpers ---
_FENCE_LINE = re.compile(r"^[ ]{0,6}(`{3,}|~{3,})(.*)$", re.M)
_COLON = re.compile(r"(?m)^:::[a-zA-Z]")
_IMPORT = re.compile(r"(?m)^\s*import\s+.*from\s+")
_JSX = re.compile(r"<[A-Z][A-Za-z0-9_]*[\s/>]")

def fence_balanced(text: str) -> bool:
    """Depth-tracked, indentation-aware, nesting-aware."""
    depth, open_char, open_len = 0, "", 0
    for line in text.splitlines():
        m = _FENCE_LINE.match(line)
        if not m:
            continue
        run = m.group(1)
        trailing = m.group(2).strip()
        if depth == 0:
            open_char, open_len, depth = run[0], len(run), 1
        elif run[0] == open_char and len(run) >= open_len and trailing == "":
            depth = 0
    return depth == 0

# --- run chunker on each ---
for r in picked:
    src = REPO_ROOT / r["cleaned_path"]
    text = src.read_text(encoding="utf-8", errors="replace")

    print("=" * 78)
    print(f"{r['library']}/{r['version']}/{r['source_path']}  [{r['format']}]")

    # cleaning sanity
    noise = {
        "colon":  len(_COLON.findall(text)),
        "import": len(_IMPORT.findall(text)),
        "jsx":    len(_JSX.findall(text)),
    }
    print(f"  cleaning noise: {noise}")

    doc = Document(
        text=text,
        metadata={
            "library":     r["library"],
            "version":     r["version"],
            "source_path": r["source_path"],
        },
    )
    doc.doc_id = make_doc_id(r["library"], r["version"], r["source_path"])

    parents, leaves = chunk_document(doc)
    print(f"  parents={len(parents)}  leaves={len(leaves)}")
    if parents:
        s = [p.metadata["token_count"] for p in parents]
        print(f"  parent tokens:  min={min(s)}  max={max(s)}")
    if leaves:
        s = [l.metadata["token_count"] for l in leaves]
        print(f"  leaf tokens:    min={min(s)}  max={max(s)}")
        print(f"  leaves w/ code: {sum(1 for l in leaves if l.metadata['has_code'])}")
        print(f"  code languages: {sorted({lang for l in leaves for lang in l.metadata['code_languages']})}")

    # --- assertions ---
    bad_fence = [l.id_ for l in leaves if not fence_balanced(l.text)]
    if bad_fence:
        print(f"  FAIL fence: {bad_fence[:3]}")
        for l in leaves:
            if l.id_ in bad_fence:
                print("  offending leaf (first 400 chars):")
                print("   ", l.text[:400].replace("\n", "\n    "))
                break
        raise AssertionError(f"unbalanced fences: {bad_fence[:3]}")

    for node in leaves:
        assert NodeRelationship.PARENT in node.relationships, node.id_
    for i, node in enumerate(leaves):
        if i > 0:
            assert NodeRelationship.PREVIOUS in node.relationships, node.id_
        if i < len(leaves) - 1:
            assert NodeRelationship.NEXT in node.relationships, node.id_

print("\nall assertions passed")

langchain/1.x/langchain/quickstart.mdx  [MDX]
  cleaning noise: {'colon': 0, 'import': 0, 'jsx': 0}
  parents=19  leaves=46
  parent tokens:  min=37  max=1225
  leaf tokens:    min=37  max=310
  leaves w/ code: 1
  code languages: ['shell']
  FAIL fence: ['a9ce472778e66b425b0efb7e7b5f2e96', 'b9be180d2a907bc55ac0fec112f6df35', 'd32d1d16852ffb7f453ea49f8b7ab748']
  offending leaf (first 400 chars):
    [Build a basic agent]
    
    ## Build a basic agent
    
    Start by creating a simple agent that can answer questions and call tools. The agent in this example uses the chosen language model, a basic weather function as a tool, and a simple prompt to guide its behavior:
    
        ```python OpenAI
        from langchain.agents import create_agent
    
        def get_weather(city: str) -> str:
            """Get weather for a give


AssertionError: unbalanced fences: ['a9ce472778e66b425b0efb7e7b5f2e96', 'b9be180d2a907bc55ac0fec112f6df35', 'd32d1d16852ffb7f453ea49f8b7ab748']

In [34]:
import json
from pathlib import Path
from llama_index.core.storage.docstore import SimpleDocumentStore

REPO_ROOT = Path.cwd().parent if Path.cwd().name in {"notebook", "notebooks"} else Path.cwd()

leaf = {
    "chunk_id": "e20a0b71e8a351b54b469e814a9af734",
    "doc_id": "00e0db8a514cde7cbcc475f6",
    "parent_chunk_id": "eb8e349199f1179639364b46ada0d059",
    "heading_path": "3. Text-to-speech > Implementation",
}

# look up the parent in the docstore
ds = SimpleDocumentStore.from_persist_dir(str(REPO_ROOT / "data/chunks/docstore"))
parent = ds.get_node(leaf["parent_chunk_id"])
print("parent found:", parent is not None)
print("  id:            ", parent.id_)
print("  level:         ", parent.metadata["level"])          # expect 0
print("  is_leaf:       ", parent.metadata["is_leaf"])        # expect False
print("  heading_path:  ", parent.metadata["heading_path"])   # expect same as leaf
print("  tokens:        ", parent.metadata["token_count"])    # expect > 562
print("  doc_id:        ", parent.relationships[list(parent.relationships)[0]].node_id)

# confirm this chunk_id appears in leaves.jsonl, not parents.jsonl
leaves_ids = {json.loads(l)["chunk_id"] for l in open(REPO_ROOT / "data/chunks/leaves.jsonl", encoding="utf-8")}
parents_ids = {json.loads(l)["chunk_id"] for l in open(REPO_ROOT / "data/chunks/parents.jsonl", encoding="utf-8")}

print()
print("this chunk is a leaf:   ", leaf["chunk_id"] in leaves_ids)
print("this chunk is a parent: ", leaf["chunk_id"] in parents_ids)
print("parent_chunk_id is a parent:", leaf["parent_chunk_id"] in parents_ids)

parent found: True
  id:             eb8e349199f1179639364b46ada0d059
  level:          0
  is_leaf:        False
  heading_path:   3. Text-to-speech > Implementation
  tokens:         848
  doc_id:         00e0db8a514cde7cbcc475f6

this chunk is a leaf:    True
this chunk is a parent:  False
parent_chunk_id is a parent: True


In [1]:
import json
from pathlib import Path
from llama_index.core.storage.docstore import SimpleDocumentStore

REPO_ROOT = Path.cwd()

ds = SimpleDocumentStore.from_persist_dir(str(REPO_ROOT / "data/chunks/docstore"))
print(f"docstore nodes: {len(ds.docs)}")   # expect 1599

# spot-check a leaf->parent link still resolves
leaf = json.loads(
    next(open(REPO_ROOT / "data/chunks/leaves.jsonl", encoding="utf-8"))
)
parent = ds.get_node(leaf["parent_chunk_id"])
print(f"leaf heading:   {leaf['metadata']['heading_path'][:60]}")
print(f"parent found:   {parent is not None}")
print(f"parent tokens:  {parent.metadata['token_count'] if parent else 'N/A'}")

FileNotFoundError: [Errno 2] No such file or directory: 'c:/Users/ASUS/OneDrive/Desktop/文件/Desktop/enterprise-rag/notebook/data/chunks/docstore/docstore.json'